# M3.7 · 时间序列交叉验证与超参优化

> 普通 K 折交叉验证会用未来预测过去，时序数据必须用 walk-forward。
> 本节实现扩展/滚动窗口切分与网格超参搜索（optuna 未装的备选）。

## 一、为什么不能随机 K 折

- 时序样本有时间顺序，随机切分会让训练集包含测试期之后的数据，造成信息泄漏；
- 正确做法：**训练永远在测试之前**，并随时间向前滚动。

| 切分 | 训练窗 | 用途 |
|---|---|---|
| 扩展窗口 walk-forward | 起点固定、终点后移 | 数据越多越稳，常用 |
| 滚动窗口 | 固定长度、整体滑动 | 适应市场结构变化、丢弃旧数据 |

**超参搜索：** 网格搜索（穷举）直观；随机/贝叶斯优化（optuna）在大空间更高效，这里用网格。


## 二、加载建模数据

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
import qlib
from qlib.data import D
from l2.config import load_universe
import config
from factor_lab2 import hfactors
from factor_lab2.combine import cross_zscore
from model_lab2 import labels, tscv
qlib.init(provider_uri=str(config.QLIB_DAILY_DIR), joblib_backend='threading')
codes = load_universe('hs300')
raw = D.features(codes, ['$open','$close','$high','$low','$volume'],
                 start_time='2018-01-01', freq='day')
op=raw['$open'].unstack('instrument'); cl=raw['$close'].unstack('instrument')
hi=raw['$high'].unstack('instrument'); lo=raw['$low'].unstack('instrument')
vo=raw['$volume'].unstack('instrument')
lab = labels.cs_rank_label(cl, 5)
hf = hfactors.build_hf_factors(op, cl, hi, lo, vo)
feat_names = ['已实现波动20','日内动量20','尾盘位置20','量价相关20','收益偏度60']
z = pd.concat({n: cross_zscore(hf[n]).stack() for n in feat_names}, axis=1)
z.index.names = ['datetime','instrument']
df = z.join(lab.stack().rename('y')).dropna().sample(60000, random_state=42).sort_index()
X = df[feat_names].values; y = df['y'].values
cut = int(len(df)*0.8)
print('样本:', df.shape)

## 三、walk-forward 切分

In [ ]:
splits = tscv.walk_forward_splits(cut, n_splits=5, train_frac=0.5)
print('扩展窗口折数:', len(splits))
for i,(tr,te) in enumerate(splits):
    print('折%d 训练[0:%d] 测试[%d:%d]' % (i, tr[-1]+1, te[0], te[-1]+1))
roll = tscv.rolling_splits(cut, train_window=10000, test_window=4000)
print('滚动窗口折数:', len(roll))

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(figsize=(10,3.6))
for i,(tr,te) in enumerate(splits):
    ax.barh(i, len(tr), left=0, color='#60a5fa')
    ax.barh(i, len(te), left=te[0], color='#f87171')
ax.set_yticks(range(len(splits)))
ax.set_xlabel('样本位置'); ax.set_title('walk-forward 切分（蓝=训练，红=测试）')
plt.tight_layout(); plt.show()

## 四、网格超参搜索

In [ ]:
from sklearn.linear_model import Ridge
def mk(alpha=1.0):
    return Ridge(alpha=alpha)
best, rows = tscv.grid_search_ts(mk, {'alpha':[1,10,100,1000]},
                                 X[:cut], y[:cut], splits)
for r in rows:
    print('alpha=%-5s 平均IC=%.4f' % (r['params']['alpha'], r['mean_score']))
print('最优:', best['params'], '平均IC=%.4f' % best['mean_score'])

In [ ]:
fig, ax = plt.subplots(figsize=(7.5,4.2))
als = [str(r['params']['alpha']) for r in rows]
scs = [r['mean_score'] for r in rows]
ax.bar(als, scs, color='#4f46e5')
ax.axhline(0, color='#333', lw=0.8)
ax.set_xlabel('Ridge alpha'); ax.set_ylabel('平均IC'); ax.set_title('超参网格搜索结果')
plt.tight_layout(); plt.show()

## 五、小结与面试要点

**面试高频：**
- 为什么时序不能随机 K 折？（答：会用未来数据，泄漏；必须训练在测试之前）
- 扩展窗口和滚动窗口区别？（答：前者起点固定用全部历史，后者固定窗长适应结构变化）
- 网格/随机/贝叶斯超参搜索怎么选？（答：小空间网格，大空间随机或贝叶斯optuna）
- 超参搜索在哪部分数据上做？（答：只在训练区间内用CV选，最终测试集只能碰一次）
- 什么是嵌套 CV？（答：外层评估、内层选超参，避免超参选择带来的乐观偏差）

## 动手练习
1. 构造 walk-forward 与滚动切分并画图；
2. 对一个模型做网格搜索选超参；
3. 说明为什么测试集只能用一次。

---
下一节 M3.8 Stacking/集成与校准。